# Experiment 1 — support collapse in the normalized-confidence readout

**Branch `readout-validity` · pre-registered in `readout_validity/PREREG.md` @ commit `8041af701965`**

**Scope.** This notebook tests whether the *normalized-confidence readout method* —
`P(True) / (P(True) + P(False))` read at a fixed position under residual-stream steering,
as used by Han, Chalmers & Izmailov (arXiv 2605.30232) — is vulnerable to **support collapse**
(probability mass leaving the `{True, False}` pair under steering). **It does not test whether
their result is wrong.** We do not have their maze-trained reward vectors and are not reproducing
them; every claim is scoped to the method, not the finding.

**How to run (phone-friendly):**
1. Runtime → Change runtime type → **GPU**. T4 works (the notebook auto-selects fp16 there —
   T4 has no native bf16 and the emulated path is ~10× slower; that is what tripped the budget
   guard on the first attempt). Expect ~45–70 min ≈ 1.5–2.2 units on T4; L4, if offered, is faster.
2. Runtime → **Run all**.
3. Read results inline. Nothing is written to Drive or to files; no authentication is requested.
   The model (Qwen3-4B-Instruct-2507, Apache-2.0, ungated) and MMLU (cais/mmlu, public) download anonymously.
4. If the session drops, per-cell progress lines already printed are the partial result —
   the run prints each (direction, α) cell as it completes, valence proxy first.
5. The final cell prints a compact **copy-pasteable JSON** summary.

**Pre-registered branches** (first match wins; full rules in PREREG.md):
gates → DISQUALIFIED (inert proxy) → B2 generic collapse → B4 collapse without ratio movement →
B1 direction-specific collapse → B3 no collapse → AMBIGUOUS.
Thresholds: collapse ≤ 0.5·S0, near-baseline ≥ 0.8·S0, separation ≥ 0.15, modulation |M| ≥ 0.10,
ratio-stable < 0.05, validity gate S0 ≥ 0.5 — all fixed before any data; sensitivity of the verdict
to threshold perturbation is reported.


In [ ]:
# Dependencies (Colab already ships torch). Quiet, no auth, public PyPI only.
%pip install -q -U "transformers>=4.51.0" "datasets>=2.19.0" accelerate


In [ ]:
# ============================== CONFIG =====================================
# All experiment-defining constants. Fixed by the pre-registration
# (readout_validity/PREREG.md); do not edit after seeing data.
import time, json, hashlib

T_START = time.time()

EXPERIMENT = "readout-validity/exp1 support collapse in normalized-confidence readout"
PREREG_COMMIT_SHA = "8041af701965057767515b1e5e07533ea0d26fae"  # readout_validity/PREREG.md on branch readout-validity

SCOPE = (
    "Tests whether the normalized-confidence readout method (P(True)/(P(True)+P(False)) "
    "at a fixed position under residual steering, as in Han, Chalmers & Izmailov, "
    "arXiv 2605.30232) is vulnerable to support collapse. It does NOT test whether their "
    "result is wrong: their maze-trained vectors are not available and are not reproduced "
    "here. Every claim is scoped to the readout method, not to their finding."
)
print(SCOPE)

# ---- budget ----------------------------------------------------------------
CONFIRMED_BUDGET = 6.0   # compute units; the preflight halts if its estimate exceeds this
BUDGET_CAP       = 6.0   # hard cap (user has 9 units; cap at 6)
MAX_RUNTIME_MIN  = 90
assert CONFIRMED_BUDGET <= BUDGET_CAP

# ---- model / steering ------------------------------------------------------
MODEL_ID  = "Qwen/Qwen3-4B-Instruct-2507"   # ungated, Apache-2.0; their primary model
LAYER_IDX = 18        # 0-indexed decoder layer (of 36); steering adds to the OUTPUT of
                      # model.model.layers[LAYER_IDX]; extraction reads hidden_states[LAYER_IDX+1]
ALPHAS      = [-4.0, -2.0, 2.0, 4.0]        # nonzero steering strengths
ALPHAS_FULL = [-4.0, -2.0, 0.0, 2.0, 4.0]
N_RANDOM_DIRS = 5

# ---- data ------------------------------------------------------------------
N_ITEMS = 500
SEED    = 20260730
MAX_PROMPT_TOKENS = 320   # turn-1 prompt cap (pre-registered filter)
BATCH_GEN, BATCH_SCORE = 32, 16

# ---- pre-registered thresholds --------------------------------------------
THRESH = dict(
    s0_gate       = 0.5,    # G1: baseline support must reach this
    collapse_frac = 0.5,    # substantial collapse: S <= 0.5*S0
    near_frac     = 0.8,    # near baseline: S >= 0.8*S0 (per individual random dir)
    sep_abs       = 0.15,   # B1 separation: min_rand S - S_val >= 0.15
    sep_sd_mult   = 3.0,    # and valence drop > max rand drop + 3*SD(rand means)
    mod_abs       = 0.10,   # modulation: |nP(+4)-nP(-4)| >= 0.10, <=1 monotonicity inversion
    ratio_stable  = 0.05,   # B4: max |nP(val,a)-nP0| < 0.05
    entropy_flag  = 4.0,    # nats; cells above are flagged low-coherence (reported, not branch-changing)
)
SENS_COLLAPSE = [0.4, 0.5, 0.6]   # verdict-sensitivity sweep (reported)
SENS_NEAR     = [0.7, 0.8, 0.9]

VERIFY_PROMPT = "Is your proposed answer correct? Answer only 'True' or 'False'."

CONFIG_HASH = hashlib.sha256(json.dumps(
    [MODEL_ID, LAYER_IDX, ALPHAS, N_RANDOM_DIRS, N_ITEMS, SEED, MAX_PROMPT_TOKENS,
     THRESH, VERIFY_PROMPT], sort_keys=True).encode()).hexdigest()
FP = PREREG_COMMIT_SHA[:8] + "/" + CONFIG_HASH[:8]   # fingerprint stamped on every record

print("fingerprint (prereg/config):", FP)
print("static a-priori estimate: ~45-70 min on T4 in fp16 (~1.4-2.2 units); the behavioral "
      "estimate with its basis is computed and enforced in the preflight below.")


In [ ]:
# ============================ PREFLIGHT ====================================
# Every check EXECUTES the thing it certifies (no string-matching of source).
import math, re, random, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers

PREFLIGHT = {}
def check(name, executed, fn):
    """Run fn(); record and print PASS/FAIL with what was actually executed."""
    try:
        info = fn()
        PREFLIGHT[name] = (True, str(info))
        print("PASS  %-16s executed: %s\n      -> %s" % (name, executed, info), flush=True)
    except Exception as e:
        PREFLIGHT[name] = (False, repr(e))
        print("FAIL  %-16s executed: %s\n      -> %r" % (name, executed, e), flush=True)

# -- 1. GPU: allocate and multiply on-device; read free/total VRAM -----------
def _gpu():
    assert torch.cuda.is_available(), "no CUDA device; set Runtime->GPU"
    x = torch.randn(256, 256, device="cuda"); y = (x @ x).sum().item()
    free, total = torch.cuda.mem_get_info()
    global GPU_NAME; GPU_NAME = torch.cuda.get_device_name(0)
    return "%s, %.1f/%.1f GB free, matmul ok (%.2f)" % (GPU_NAME, free/2**30, total/2**30, y)
check("gpu", "torch matmul on cuda + mem_get_info", _gpu)

# -- 2. Model: download, load, run a real generation -------------------------
# Dtype is chosen by hardware: pre-Ampere GPUs (T4 = sm_75) have NO native bf16;
# torch "supports" bf16 there via emulation at ~10x cost (measured 2026-07-30 on a
# Colab T4: ~18 s/batch bf16 vs ~2 s expected -> 11.67-unit estimate, budget halt).
# fp16 uses the T4 tensor cores; the readout softmax is fp32 either way.
def _model():
    global model, tok, DTYPE
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    cap = torch.cuda.get_device_capability(0)
    try:
        bf16_native = torch.cuda.is_bf16_supported(including_emulation=False)
    except TypeError:
        bf16_native = cap[0] >= 8
    DTYPE = torch.bfloat16 if bf16_native else torch.float16
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE, device_map="cuda")
    except TypeError:  # transformers < 5 uses torch_dtype
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="cuda")
    model.eval()
    nl = model.config.num_hidden_layers
    assert LAYER_IDX < nl, "LAYER_IDX %d >= n_layers %d" % (LAYER_IDX, nl)
    ii = tok("2+2=", return_tensors="pt").to("cuda")
    out = tok.decode(model.generate(**ii, max_new_tokens=5, do_sample=False)[0][ii.input_ids.shape[1]:])
    return "transformers %s, %d layers, d=%d, sm_%d%d native-bf16=%s -> dtype=%s, gen('2+2=')=%r" % (
        transformers.__version__, nl, model.config.hidden_size, cap[0], cap[1],
        bf16_native, DTYPE, out.strip()[:20])
check("model", "download+load %s (hw-selected dtype), greedy 5-token generation" % MODEL_ID, _model)

# -- 3. Chat template: build the exact 3-message structure, audit the tail ---
def _template():
    msgs = [{"role": "user", "content": "Q?"},
            {"role": "assistant", "content": "A."},
            {"role": "user", "content": VERIFY_PROMPT}]
    s = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    assert s.count("<|im_start|>assistant") >= 2, "template lacks assistant headers"
    assert "<think>" not in s, "template inserts a <think> block; read position would shift"
    ids = tok(s, add_special_tokens=False).input_ids
    tail = [tok.decode([t]) for t in ids[-5:]]
    return "read position follows tail tokens %r" % (tail,)
check("template", "apply_chat_template on 2-turn structure; tail-token audit", _template)

# -- 4. Token variants: 'True'/'False' with and without leading space --------
def _variants():
    global CAND_IDS
    CAND_IDS = {}
    for v in ["True", " True", "False", " False"]:
        e = tok(v, add_special_tokens=False).input_ids
        assert len(e) == 1, "%r is not a single token: %r" % (v, e)
        CAND_IDS[v] = e[0]
    return "single-token ids " + str(CAND_IDS) + " (G2 pass)"
check("variants", "encode 4 True/False variants, assert single-token", _variants)

# -- 5. MMLU: download public test split, parse a real item ------------------
def _mmlu():
    global MMLU_RAW
    from datasets import load_dataset
    ds = load_dataset("cais/mmlu", "all", split="test")
    MMLU_RAW = [r for r in ds if r["subject"].startswith("high_school")]
    r = MMLU_RAW[0]
    assert len(r["choices"]) == 4 and isinstance(r["answer"], int)
    return "%d high-school items (of %d total), sample subject=%s" % (
        len(MMLU_RAW), len(ds), r["subject"])
check("mmlu", "datasets.load_dataset('cais/mmlu','all',test) + parse item", _mmlu)

# -- 6. Scoring path: resolve last-position-only logits kwarg behaviorally ---
def _score_path():
    global SCORE_MODE
    ids = torch.randint(10, 1000, (4, 64), device="cuda")
    attn = torch.ones_like(ids); pos = (attn.cumsum(-1) - 1)
    SCORE_MODE = "full"
    for kw in ["logits_to_keep", "num_logits_to_keep"]:
        try:
            with torch.no_grad():
                o = model(input_ids=ids, attention_mask=attn, position_ids=pos,
                          use_cache=False, **{kw: 1})
            assert o.logits.shape[1] == 1
            SCORE_MODE = kw
            break
        except (TypeError, AssertionError):
            continue
    if SCORE_MODE == "full":
        with torch.no_grad():
            o = model(input_ids=ids, attention_mask=attn, position_ids=pos, use_cache=False)
        assert o.logits.shape[1] == 64
    assert torch.isfinite(o.logits).all(), "non-finite logits under %s" % DTYPE
    return "mode=%s, logits shape verified %s, finite under %s" % (
        SCORE_MODE, tuple(o.logits.shape), DTYPE)
check("score_path", "forward pass probing logits_to_keep support", _score_path)

def model_forward(ids, attn, pos):
    kw = {} if SCORE_MODE == "full" else {SCORE_MODE: 1}
    return model(input_ids=ids, attention_mask=attn, position_ids=pos, use_cache=False, **kw)

# -- 7. Budget: TIME the real scoring op, extrapolate, enforce ---------------
RATES = {"T4": 1.84, "L4": 4.82, "A100": 13.08, "V100": 4.91}
RATE_BASIS = ("approximate Colab burn rates in units/hr as commonly reported 2025-2026; "
              "the rate shown in Colab's own Resources panel governs. Unknown GPUs are "
              "billed here at the A100 rate as a conservative bound.")
def _budget():
    global EST_MIN, EST_UNITS, GPU_RATE
    ids = torch.randint(10, 1000, (BATCH_SCORE, 360), device="cuda")
    attn = torch.ones_like(ids); pos = (attn.cumsum(-1) - 1)
    with torch.no_grad():
        model_forward(ids, attn, pos); torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(3):
            model_forward(ids, attn, pos)
        torch.cuda.synchronize()
    per_batch = (time.time() - t0) / 3
    n_cells = 1 + (2 + N_RANDOM_DIRS) * len(ALPHAS)            # baseline + 7 dirs x 4 alphas
    n_batches = math.ceil(N_ITEMS / BATCH_SCORE) * n_cells
    score_min = per_batch * n_batches / 60
    gen_min   = per_batch * 8 * math.ceil(N_ITEMS / BATCH_GEN) / 60   # 12-token greedy: prefill + decode steps
    misc_min  = 4.0                                             # directions, tables, figure
    elapsed_min = (time.time() - T_START) / 60
    EST_MIN = (elapsed_min + score_min + gen_min + misc_min) * 1.15   # +15% safety margin
    GPU_RATE = next((v for k, v in RATES.items() if k in GPU_NAME), RATES["A100"])
    EST_UNITS = GPU_RATE * EST_MIN / 60
    msg = ("%.2f s/batch -> scoring %.0f min + gen %.0f min + setup %.0f min + misc %.0f min "
           "-> %.0f min total (incl. 15%% margin); %.2f units at %.2f units/hr. Basis: %s" %
           (per_batch, score_min, gen_min, elapsed_min, misc_min, EST_MIN, EST_UNITS,
            GPU_RATE, RATE_BASIS))
    hint = (" HINT: do NOT raise CONFIRMED_BUDGET; if this trips, the per-batch time is the "
            "bottleneck — switch Runtime->Change runtime type to L4 (best units/throughput "
            "here) and rerun.")
    assert EST_UNITS <= CONFIRMED_BUDGET, "estimate %.2f units exceeds CONFIRMED_BUDGET %.1f — halting. %s%s" % (EST_UNITS, CONFIRMED_BUDGET, msg, hint)
    assert EST_MIN <= MAX_RUNTIME_MIN, "estimate %.0f min exceeds %d min cap — halting. %s%s" % (EST_MIN, MAX_RUNTIME_MIN, msg, hint)
    return msg
check("budget", "3 timed batched forwards at production shape, extrapolated", _budget)

failed = [k for k, (ok, _) in PREFLIGHT.items() if not ok]
assert not failed, "PREFLIGHT FAILED: %s — do not proceed" % failed
print("\nPREFLIGHT COMPLETE — all %d checks executed their targets and passed." % len(PREFLIGHT))


In [ ]:
# ============================ DATA PREP ====================================
# Subsample N_ITEMS MMLU high-school questions (seeded), keep prompts <= cap.
from collections import Counter

def mk_prompt(r):
    c = r["choices"]
    return ("Answer the following multiple-choice question. Respond with only the "
            "letter of the correct answer.\n\n%s\n\nA. %s\nB. %s\nC. %s\nD. %s"
            % (r["question"], c[0], c[1], c[2], c[3]))

rng = random.Random(SEED)
pool = list(MMLU_RAW); rng.shuffle(pool)
ITEMS = []
for r in pool:
    p = mk_prompt(r)
    n = len(tok.apply_chat_template([{"role": "user", "content": p}],
                                    tokenize=True, add_generation_prompt=True))
    if n <= MAX_PROMPT_TOKENS:
        ITEMS.append(dict(prompt=p, subject=r["subject"], answer_idx=int(r["answer"])))
    if len(ITEMS) == N_ITEMS:
        break
assert len(ITEMS) == N_ITEMS, "only %d items under token cap" % len(ITEMS)
print("%d items selected (turn-1 prompt <= %d tokens), seed %d" % (len(ITEMS), MAX_PROMPT_TOKENS, SEED))
print("subjects:", dict(Counter(i["subject"] for i in ITEMS)))


In [ ]:
# ==================== TURN 1: unsteered answers (greedy) ===================
# "Sample one unsteered answer per question" — implemented as greedy decoding
# (deterministic), pre-registered. No steering hook exists yet.
t0 = time.time()
LETTER_RE = re.compile(r"[ABCD]")
n_parsed = n_correct = 0
for b0 in range(0, len(ITEMS), BATCH_GEN):
    batch = ITEMS[b0:b0 + BATCH_GEN]
    texts = [tok.apply_chat_template([{"role": "user", "content": it["prompt"]}],
                                     tokenize=False, add_generation_prompt=True) for it in batch]
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=12, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    for it, row in zip(batch, out[:, enc.input_ids.shape[1]:]):
        ans = tok.decode(row, skip_special_tokens=True).strip().split("\n")[0].strip()
        it["answer_text"] = ans if ans else "?"
        m = LETTER_RE.search(ans)
        it["letter"] = m.group(0) if m else None
        if m: n_parsed += 1
        if m and "ABCD"[it["answer_idx"]] == m.group(0): n_correct += 1
    if (b0 // BATCH_GEN) % 4 == 0:
        print("  gen %d/%d  %.0fs" % (b0 + len(batch), len(ITEMS), time.time() - t0), flush=True)
ACC_T1 = n_correct / len(ITEMS)
print("turn-1 done in %.0fs: letter parsed %d/%d, accuracy %.3f"
      % (time.time() - t0, n_parsed, len(ITEMS), ACC_T1))


In [ ]:
# ============ TURN 2: sequences + assistant-token steering masks ===========
# Steering applies at assistant-turn tokens only: the turn-1 assistant span
# (header..im_end) and the final assistant generation header, which includes
# the read position (last token).
IM_A, IM_END = "<|im_start|>assistant", "<|im_end|>"

def build_item(it):
    msgs = [{"role": "user", "content": it["prompt"]},
            {"role": "assistant", "content": it["answer_text"]},
            {"role": "user", "content": VERIFY_PROMPT}]
    full = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    i1 = full.find(IM_A); e1 = full.find(IM_END, i1) + len(IM_END)
    i2 = full.rfind(IM_A)
    assert 0 <= i1 < e1 <= i2, "assistant spans not found/ordered"
    spans = [(i1, e1), (i2, len(full))]
    enc = tok(full, add_special_tokens=False, return_offsets_mapping=True)
    ids, offs = enc["input_ids"], enc["offset_mapping"]
    mask = [any(ts < b and te > a for a, b in spans) for (ts, te) in offs]
    assert mask[-1], "read position not masked"
    assert sum(mask) >= 4, "assistant span mask too small"
    it["ids"], it["amask"] = ids, mask
    return len(ids), sum(mask)

lens, msks = zip(*[build_item(it) for it in ITEMS])
# behavioral consistency: offsets-encode must equal template's own tokenization
ref = tok.apply_chat_template([{"role": "user", "content": ITEMS[0]["prompt"]},
                               {"role": "assistant", "content": ITEMS[0]["answer_text"]},
                               {"role": "user", "content": VERIFY_PROMPT}],
                              tokenize=True, add_generation_prompt=True)
assert list(ref) == list(ITEMS[0]["ids"]), "offset-mapping tokenization mismatch"
print("turn-2 sequences built: len median %d max %d; assistant-masked tokens median %d; "
      "read position masked for all %d items" % (sorted(lens)[len(lens)//2], max(lens),
      sorted(msks)[len(msks)//2], len(ITEMS)))


In [ ]:
# ==================== DIRECTIONS (difference of means) =====================
# All directions rescaled to the norm of the raw valence vector (matched-norm).
# Sign: +alpha pushes toward FAILURE (negative valence) / FORMAL register.
FAILURE_TEXTS = [
 "I failed the task completely.", "My attempt was a total failure.",
 "I could not solve the problem no matter what I tried.",
 "Everything I did went wrong and the task collapsed.",
 "I made mistake after mistake and finally gave up.",
 "The project fell apart; I did not accomplish anything.",
 "I was unable to finish; it was hopeless.",
 "My answer was wrong and the whole effort was wasted.",
 "I botched the assignment badly.", "No matter how hard I worked, I kept failing.",
 "The result was a disaster; nothing worked.",
 "I lost, and all my effort came to nothing.",
 "It was impossible; I failed at every step.",
 "My performance was terrible and the task ended in failure.",
 "I broke everything I touched and achieved nothing.",
 "The attempt ended in complete defeat."]
SUCCESS_TEXTS = [
 "I completed the task perfectly.", "My attempt was a total success.",
 "I solved the problem easily on the first try.",
 "Everything I did went right and the task came together.",
 "I made steady progress and finally finished.",
 "The project came together; I accomplished everything.",
 "I was able to finish; it was straightforward.",
 "My answer was right and the whole effort paid off.",
 "I aced the assignment.", "The harder I worked, the more I succeeded.",
 "The result was excellent; everything worked.",
 "I won, and all my effort paid off.",
 "It was easy; I succeeded at every step.",
 "My performance was superb and the task ended in success.",
 "Everything I touched improved and I achieved my goals.",
 "The attempt ended in complete victory."]
FORMAL_TEXTS = [
 "Dear Dr. Reynolds, I am writing to request an extension of the submission deadline.",
 "The committee hereby approves the proposed amendment to the charter.",
 "Pursuant to our agreement, the undersigned shall deliver the report by Friday.",
 "We respectfully request that you review the enclosed documentation.",
 "It is with great pleasure that I accept your kind invitation.",
 "The board convened to deliberate upon the quarterly financial statements.",
 "Please find attached the minutes of the previous meeting.",
 "I would be most grateful if you could confirm receipt of this letter.",
 "The applicant possesses the requisite qualifications for the position.",
 "We regret to inform you that the proposal has been declined.",
 "Kindly direct all further correspondence to the undersigned.",
 "The institution maintains the highest standards of academic integrity.",
 "Your prompt attention to this matter would be greatly appreciated.",
 "The parties agree to resolve any disputes through binding arbitration.",
 "I hereby certify that the foregoing statements are true and correct.",
 "The ceremony will commence promptly at seven o'clock in the evening."]
INFORMAL_TEXTS = [
 "hey dude can u push the deadline back a bit lol",
 "yeah the gang said ok to changing the rules haha",
 "so like I'll get you the thing by friday np",
 "check this stuff out when u get a sec",
 "omg yes I'm so down for the party",
 "the guys got together to chat about the money stuff",
 "here's the notes from last time btw",
 "lemme know u got this ok?",
 "tbh she's totally got the chops for the gig",
 "sorry bro, gonna pass on your idea",
 "just text me directly next time k?",
 "this school is legit about not cheating fr",
 "hit me back asap plz",
 "if we fight about it we'll just flip a coin lol",
 "swear all this is legit no cap",
 "party starts at 7 don't be late!!"]

@torch.no_grad()
def mean_resid(texts):
    """Per-text mean over real tokens of hidden_states[LAYER_IDX+1] (= output of layers[LAYER_IDX])."""
    outs = []
    for b0 in range(0, len(texts), 16):
        enc = tok(texts[b0:b0+16], return_tensors="pt", padding=True,
                  add_special_tokens=False).to("cuda")
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states[LAYER_IDX + 1]
        m = enc.attention_mask.unsqueeze(-1).to(hs.dtype)
        outs.append(((hs * m).sum(1) / m.sum(1)).float().cpu())
    return torch.cat(outs)

v_val_raw = mean_resid(FAILURE_TEXTS).mean(0) - mean_resid(SUCCESS_TEXTS).mean(0)
NORM_REF = v_val_raw.norm().item()
v_reg_raw = mean_resid(FORMAL_TEXTS).mean(0) - mean_resid(INFORMAL_TEXTS).mean(0)
V = {"valence": v_val_raw.clone()}
V["register"] = v_reg_raw * (NORM_REF / v_reg_raw.norm())
for i in range(N_RANDOM_DIRS):
    g = torch.Generator().manual_seed(1000 + i)
    r = torch.randn(model.config.hidden_size, generator=g)
    V["rand%d" % i] = r * (NORM_REF / r.norm())

# scale context: median residual norm at the steering site on real task tokens
@torch.no_grad()
def median_resid_norm():
    ns = []
    for b0 in range(0, 32, BATCH_SCORE):
        sub = ITEMS[b0:b0 + BATCH_SCORE]
        enc = tok.pad({"input_ids": [it["ids"] for it in sub]}, return_tensors="pt").to("cuda")
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states[LAYER_IDX + 1]
        m = enc.attention_mask.bool()
        ns.append(hs[m].float().norm(dim=-1).cpu())
    return torch.cat(ns).median().item()
MED_RESID = median_resid_norm()

print("matched norm ||v|| = %.2f; median residual norm at layer %d = %.2f" % (NORM_REF, LAYER_IDX, MED_RESID))
for a in ALPHAS:
    print("  |alpha|=%.0f -> ||alpha*v|| / median resid = %.3f" % (abs(a), abs(a) * NORM_REF / MED_RESID))
names = list(V)
print("pairwise cosines:")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        c = torch.nn.functional.cosine_similarity(V[names[i]], V[names[j]], dim=0).item()
        if abs(c) > 0.05 or j == i + 1:
            print("  cos(%s,%s)=%.3f" % (names[i], names[j], c))


In [ ]:
# ==================== STEERING + SCORING MACHINERY =========================
class SteerHook:
    """Adds alpha*v to the output of one decoder layer at masked positions."""
    def __init__(self, module):
        self.vec = None; self.mask = None
        self.h = module.register_forward_hook(self._fn)
    def set(self, vec, alpha, mask):
        self.vec = None if (vec is None or alpha == 0) else (alpha * vec).to("cuda")
        self.mask = mask
    def clear(self):
        self.vec = None; self.mask = None
    def _fn(self, mod, inp, out):
        if self.vec is None:
            return out
        hs = out[0] if isinstance(out, tuple) else out
        hs = hs + self.mask.unsqueeze(-1).to(hs.dtype) * self.vec.to(hs.dtype)
        return (hs,) + tuple(out[1:]) if isinstance(out, tuple) else hs

HOOK = SteerHook(model.model.layers[LAYER_IDX])

def pad_left(sub):
    L = max(len(it["ids"]) for it in sub)
    ids  = torch.full((len(sub), L), tok.pad_token_id, dtype=torch.long)
    attn = torch.zeros((len(sub), L), dtype=torch.long)
    am   = torch.zeros((len(sub), L), dtype=torch.bool)
    for i, it in enumerate(sub):
        n = len(it["ids"])
        ids[i, L-n:] = torch.tensor(it["ids"]); attn[i, L-n:] = 1
        am[i, L-n:] = torch.tensor(it["amask"])
    pos = (attn.cumsum(-1) - 1).clamp(min=0)
    return (ids.cuda(), attn.cuda(), am.cuda(), pos.cuda())

@torch.no_grad()
def run_condition(direction, vec, alpha):
    """Score every item at the read position. Returns raw records (variant-agnostic)."""
    recs = []
    for b0 in range(0, len(ITEMS), BATCH_SCORE):
        sub = ITEMS[b0:b0 + BATCH_SCORE]
        ids, attn, am, pos = pad_left(sub)
        HOOK.set(vec, alpha, am)
        try:
            logits = model_forward(ids, attn, pos).logits[:, -1, :].float()
        finally:
            HOOK.clear()
        probs = torch.softmax(logits, -1)
        ent = -(probs * (probs + 1e-12).log()).sum(-1)
        tp, ti = probs.topk(5, -1)
        cand = {v: probs[:, cid] for v, cid in CAND_IDS.items()}
        for i in range(len(sub)):
            recs.append(dict(
                fp=FP, item=b0 + i, direction=direction, alpha=alpha,
                p_cand={v: cand[v][i].item() for v in CAND_IDS},
                entropy=ent[i].item(),
                top5=[(tok.decode([ti[i, k].item()]), tp[i, k].item()) for k in range(5)]))
    return recs

def finalize(rec):
    """Apply the frozen token-variant choice; compute pre-registered metrics."""
    pt, pf = rec["p_cand"][TRUE_KEY], rec["p_cand"][FALSE_KEY]
    s = pt + pf
    rec.update(p_true=pt, p_false=pf, support=s,
               np_true=(pt / s) if s > 0 else float("nan"))
    return rec

def summarize(recs):
    import statistics as st
    S  = st.mean(r["support"] for r in recs)
    nps = [r["np_true"] for r in recs if r["support"] > 0]
    top1 = Counter(r["top5"][0][0] for r in recs).most_common(1)[0]
    return dict(S=S, nP=st.mean(nps) if nps else float("nan"),
                p_true=st.mean(r["p_true"] for r in recs),
                p_false=st.mean(r["p_false"] for r in recs),
                ent_med=st.median(r["entropy"] for r in recs),
                top1_mode=top1[0], top1_frac=top1[1] / len(recs),
                n_zero_support=sum(1 for r in recs if r["support"] == 0))
print("machinery ready; hook installed on model.model.layers[%d]" % LAYER_IDX)


In [ ]:
# ================ BASELINE (alpha = 0): token ids + S0, BEFORE steering ====
import statistics as st
t0 = time.time()
recs0 = run_condition("baseline", None, 0.0)

mass_ns = st.mean(r["p_cand"]["True"] + r["p_cand"]["False"] for r in recs0)
mass_sp = st.mean(r["p_cand"][" True"] + r["p_cand"][" False"] for r in recs0)
if mass_ns >= mass_sp:
    TRUE_KEY, FALSE_KEY = "True", "False"
else:
    TRUE_KEY, FALSE_KEY = " True", " False"
TRUE_ID, FALSE_ID = CAND_IDS[TRUE_KEY], CAND_IDS[FALSE_KEY]
for r in recs0:
    finalize(r)
BASE = summarize(recs0)
S0, NP0 = BASE["S"], BASE["nP"]

print("scored token ids (frozen for the whole run):")
print("  TRUE  = %r -> id %d      FALSE = %r -> id %d" % (TRUE_KEY, TRUE_ID, FALSE_KEY, FALSE_ID))
print("  variant-pair baseline mass: no-space %.4f vs space %.4f -> chose %s"
      % (mass_ns, mass_sp, "no-space" if mass_ns >= mass_sp else "space"))
print("BASELINE (unsteered, n=%d):  S0 = %.4f   nP0 = %.4f   median entropy %.3f nats  (%.0fs)"
      % (len(ITEMS), S0, NP0, BASE["ent_med"], time.time() - t0))
print("sample top-5 at read position (3 items):")
for r in recs0[:3]:
    print("   ", [(t_, round(p_, 4)) for t_, p_ in r["top5"]])

# Validity gates (pre-registered): fail -> stop, no branch is evaluated.
assert S0 >= THRESH["s0_gate"], (
    "GATE G1 FAILED: S0=%.3f < %.2f — the model is not concentrating mass on the scored "
    "pair even unsteered; SETUP INVALID, stopping before any steering." % (S0, THRESH["s0_gate"]))
print("GATE G1 pass (S0 >= %.2f); GATE G2 passed in preflight. Steering may proceed." % THRESH["s0_gate"])


In [ ]:
# ========================= MAIN LOOP: 7 x 4 cells ==========================
# Valence first (the disqualifier hinges on it), then 5 random, then register.
# Each cell prints as it completes — partial output IS the partial result.
RESULTS, RECORDS = {}, []
for name in list(V):
    RESULTS[(name, 0.0)] = BASE          # shared unsteered baseline
DIR_ORDER = ["valence"] + ["rand%d" % i for i in range(N_RANDOM_DIRS)] + ["register"]
n_cells = len(DIR_ORDER) * len(ALPHAS); ci = 0
t_loop = time.time()
for name in DIR_ORDER:
    for a in ALPHAS:
        tc = time.time()
        recs = [finalize(r) for r in run_condition(name, V[name], a)]
        summ = summarize(recs)
        RESULTS[(name, a)] = summ; RECORDS += recs; ci += 1
        print("[%2d/%d] dir=%-8s a=%+3.0f  S=%.3f (S0=%.3f)  nP=%.3f  ent_med=%.2f  "
              "top1=%r(%.2f)  %3.0fs  elapsed %4.0fs"
              % (ci, n_cells, name, a, summ["S"], S0, summ["nP"], summ["ent_med"],
                 summ["top1_mode"], summ["top1_frac"], time.time() - tc,
                 time.time() - T_START), flush=True)
    if name == "valence":
        M_int = RESULTS[("valence", 4.0)]["nP"] - RESULTS[("valence", -4.0)]["nP"]
        Smin = min(RESULTS[("valence", a)]["S"] for a in ALPHAS)
        print(" -- interim (valence done): modulation M=nP(+4)-nP(-4)=%+.3f; "
              "min S = %.3f = %.2f*S0" % (M_int, Smin, Smin / S0), flush=True)
        worst_a = min(ALPHAS, key=lambda a: RESULTS[("valence", a)]["S"])
        print(" -- what eats the mass at alpha=%+.0f (3 items):" % worst_a)
        vr = [r for r in RECORDS if r["direction"] == "valence" and r["alpha"] == worst_a]
        for r in vr[:3]:
            print("     ", [(t_, round(p_, 3)) for t_, p_ in r["top5"]])
print("main loop done in %.1f min; total elapsed %.1f min"
      % ((time.time() - t_loop) / 60, (time.time() - T_START) / 60))


In [ ]:
# ===================== PER-DIRECTION TABLES (never pooled) =================
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.float_format", lambda x: "%.3f" % x)
rows = []
for (name, a), s in sorted(RESULTS.items()):
    rows.append(dict(direction=name, alpha=a, S=s["S"], S_over_S0=s["S"] / S0,
                     nP=s["nP"], p_true=s["p_true"], p_false=s["p_false"],
                     ent_med=s["ent_med"], top1=s["top1_mode"], top1_frac=s["top1_frac"],
                     low_coherence=s["ent_med"] > THRESH["entropy_flag"]))
DF = pd.DataFrame(rows)
print("== support mass S (rows=direction, cols=alpha) ==")
print(DF.pivot(index="direction", columns="alpha", values="S"))
print("\n== normalized P(True) ==")
print(DF.pivot(index="direction", columns="alpha", values="nP"))
print("\n== median next-token entropy (nats; flag > %.1f) ==" % THRESH["entropy_flag"])
print(DF.pivot(index="direction", columns="alpha", values="ent_med"))
print("\n== modal top-1 token per cell ==")
print(DF.pivot(index="direction", columns="alpha", values="top1"))
flags = DF[DF.low_coherence]
if len(flags):
    print("\nLOW-COHERENCE cells (median entropy > %.1f nats) — collapse there may reflect "
          "general incoherence, not targeted mass movement:" % THRESH["entropy_flag"])
    print(flags[["direction", "alpha", "S", "ent_med"]].to_string(index=False))
else:
    print("\nno low-coherence cells (all median entropies <= %.1f nats)" % THRESH["entropy_flag"])


In [ ]:
# ================= PRE-REGISTERED BRANCH EVALUATION ========================
# Rules and order exactly as in readout_validity/PREREG.md @ 8041af70.
import statistics as st
RANDS = ["rand%d" % i for i in range(N_RANDOM_DIRS)]

def evaluate(collapse_frac, near_frac):
    S  = {k: v["S"]  for k, v in RESULTS.items()}
    nP = {k: v["nP"] for k, v in RESULTS.items()}
    coll = lambda d, a: S[(d, a)] <= collapse_frac * S0
    near = lambda d, a: S[(d, a)] >= near_frac * S0
    det = {}
    # modulation (fixed thresholds)
    M = nP[("valence", 4.0)] - nP[("valence", -4.0)]
    curve = [nP[("valence", a)] for a in ALPHAS_FULL]
    diffs = [curve[i + 1] - curve[i] for i in range(4)]
    inversions = sum(1 for d in diffs if d * M < 0)
    mod = abs(M) >= THRESH["mod_abs"] and inversions <= 1
    det.update(M=M, inversions=inversions, modulation_present=mod)
    # disqualifier: inert proxy (support-movement bound fixed at 0.8 per prereg)
    val_moved = any(S[("valence", a)] <= 0.8 * S0 for a in ALPHAS)
    det["val_moved_support"] = val_moved
    if not mod and not val_moved:
        return "DISQUALIFIED", det
    # B2 generic collapse
    for a in ALPHAS:
        n_coll = sum(coll(r, a) for r in RANDS)
        if n_coll >= 3:
            det["B2_alpha"] = a; det["B2_n_rand_collapsed"] = n_coll
            return "B2", det
    # B4 collapse without ratio movement
    ratio_dev = max(abs(nP[("valence", a)] - NP0) for a in ALPHAS)
    det["max_ratio_dev"] = ratio_dev
    val_coll_alphas = [a for a in ALPHAS if coll("valence", a)]
    if val_coll_alphas and ratio_dev < THRESH["ratio_stable"]:
        det["B4_alphas"] = val_coll_alphas
        return "B4", det
    # B1 direction-specific collapse
    for a in val_coll_alphas:
        if all(near(r, a) for r in RANDS):
            rand_means = [S[(r, a)] for r in RANDS]
            sep1 = min(rand_means) - S[("valence", a)] >= THRESH["sep_abs"]
            sep2 = (S0 - S[("valence", a)]) > (S0 - min(rand_means)) + \
                   THRESH["sep_sd_mult"] * st.stdev(rand_means)
            det["B1_alpha"] = a
            det["B1_sep_abs"] = min(rand_means) - S[("valence", a)]
            det["B1_sep_pass"] = bool(sep1 and sep2)
            if sep1 and sep2:
                det["register_also_collapsed"] = any(coll("register", aa) for aa in ALPHAS)
                return "B1", det
    # B3 no collapse
    if all(S[(d, a)] >= near_frac * S0 for d in list(V) for a in ALPHAS):
        return "B3", det
    return "AMBIGUOUS", det

VERDICT, DETAIL = evaluate(THRESH["collapse_frac"], THRESH["near_frac"])
GRID = {"%.1f/%.1f" % (cf, nf): evaluate(cf, nf)[0] for cf in SENS_COLLAPSE for nf in SENS_NEAR}
FRAGILE = len(set(GRID.values())) > 1

MEANING = {
 "DISQUALIFIED": ("The valence proxy moved neither the ratio nor the support: it is not a "
                  "stand-in for their vector, and this run is UNINFORMATIVE about their readout. "
                  "Terminal; do not reinterpret."),
 "B2": ("GENERIC collapse: random matched-norm directions also collapse support. Their flat "
        "controls are then positive evidence AGAINST the artifact story. Report as such; stop."),
 "B4": ("Collapse WITHOUT ratio movement: support falls but normalized P(True) is stable — the "
        "normalization is doing its job. Weaker finding than B1; report as B4 only."),
 "B1": ("DIRECTION-SPECIFIC collapse: the readout is vulnerable specifically to directions that "
        "promote competing tokens, while matched-norm random controls hold — consistent with why "
        "flat controls would not reveal it. Scoped to the METHOD, not to their finding."),
 "B3": ("NO collapse under a modulating proxy: the readout survives this test. This is a real "
        "result, not a failure — it strengthens their method."),
 "AMBIGUOUS": ("No pre-registered pattern matched. Report the numbers; claim no branch."),
}
print("=" * 70)
print("PRE-REGISTERED VERDICT: %s" % VERDICT)
print(MEANING[VERDICT])
if VERDICT == "B1" and DETAIL.get("register_also_collapsed"):
    print("NOTE: the register direction ALSO collapsed -> per prereg the claim wording weakens "
          "from 'valence-specific' to 'semantic-direction-specific'.")
print("detail:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in DETAIL.items()})
print("threshold sensitivity (collapse_frac/near_frac -> verdict):", GRID)
print("verdict is %s across the pre-registered sensitivity grid."
      % ("FRAGILE — flips" if FRAGILE else "stable"))


In [ ]:
# ============================ FIGURE (inline) ==============================
import matplotlib.pyplot as plt

COL_VAL, COL_REG, COL_RAND, COL_REF = "#2a78d6", "#1baf7a", "#52514e", "#b9b8b3"
MARKS = ["o", "s", "^", "D", "v"]
fig, axes = plt.subplots(2, 3, figsize=(12, 6.5), dpi=110, sharex=True)
xs = ALPHAS_FULL

def series(name, metric):
    return [RESULTS[(name, a)][metric] for a in xs]

for row, (metric, ylab, ref) in enumerate([
        ("S", "support mass  P(True)+P(False)", S0),
        ("nP", "normalized P(True)", NP0)]):
    for col, panel in enumerate(["valence", "rand", "register"]):
        ax = axes[row][col]
        ax.axhline(ref, color=COL_REF, lw=1, ls="--", zorder=1)
        if panel == "rand":
            for i in range(N_RANDOM_DIRS):
                ax.plot(xs, series("rand%d" % i, metric), color=COL_RAND, lw=1.4,
                        marker=MARKS[i], ms=5, alpha=0.85, label="seed %d" % i)
            if row == 0:
                ax.legend(fontsize=7, frameon=False, ncol=2, loc="lower left")
        else:
            c = COL_VAL if panel == "valence" else COL_REG
            ax.plot(xs, series(panel, metric), color=c, lw=2, marker="o", ms=6)
        ax.set_ylim(0, 1.05)
        ax.grid(axis="y", alpha=0.25)
        for s in ["top", "right"]:
            ax.spines[s].set_visible(False)
        if row == 0:
            ax.set_title({"valence": "Valence proxy (failure − success)",
                          "rand": "Matched-norm random (5, individual)",
                          "register": "Register (formal − informal)"}[panel], fontsize=10)
        if row == 1:
            ax.set_xlabel("steering strength α")
        if col == 0:
            ax.set_ylabel(ylab, fontsize=9)
fig.suptitle("Support collapse test — %s, layer %d, n=%d  (dashed = unsteered baseline)"
             % (MODEL_ID.split("/")[-1], LAYER_IDX, len(ITEMS)), fontsize=11)
fig.tight_layout()
plt.show()


In [ ]:
# ==================== COPY-PASTEABLE JSON SUMMARY ==========================
CODE_HASH = hashlib.sha256("\n<CELL>\n".join(In).encode()).hexdigest()
ELAPSED_MIN = (time.time() - T_START) / 60
r4 = lambda x: (round(x, 4) if isinstance(x, float) else x)

summary = {
  "experiment": EXPERIMENT,
  "fingerprint": {"prereg_commit": PREREG_COMMIT_SHA, "config_hash": CONFIG_HASH[:16],
                   "executed_code_sha256": CODE_HASH[:16]},
  "model": MODEL_ID, "dtype": str(DTYPE), "layer_idx": LAYER_IDX,
  "n_items": len(ITEMS), "seed": SEED,
  "gpu": GPU_NAME, "elapsed_min": round(ELAPSED_MIN, 1),
  "units_spent_est": round(GPU_RATE * ELAPSED_MIN / 60, 2),
  "budget": {"confirmed": CONFIRMED_BUDGET, "preflight_est_units": round(EST_UNITS, 2),
              "rate_basis": "see preflight output"},
  "gates": {"G1_S0": r4(S0), "G1_pass": True, "G2_single_token_variants": True},
  "token_ids": {"true_key": TRUE_KEY, "true_id": TRUE_ID,
                 "false_key": FALSE_KEY, "false_id": FALSE_ID},
  "baseline": {"S0": r4(S0), "nP0": r4(NP0), "ent_med": r4(BASE["ent_med"]),
                "turn1_accuracy": r4(ACC_T1)},
  "norm_context": {"vec_norm": r4(NORM_REF), "median_resid_norm_L%d" % LAYER_IDX: r4(MED_RESID),
                    "alpha4_over_resid": r4(4 * NORM_REF / MED_RESID)},
  "verdict": VERDICT,
  "verdict_detail": {k: r4(v) for k, v in DETAIL.items()},
  "sensitivity": {"fragile": FRAGILE, "grid": GRID},
  "cells": [{"dir": name, "alpha": a, "S": r4(s["S"]), "S_over_S0": r4(s["S"] / S0),
              "nP": r4(s["nP"]), "p_true": r4(s["p_true"]), "p_false": r4(s["p_false"]),
              "ent_med": r4(s["ent_med"]), "top1": s["top1_mode"],
              "top1_frac": r4(s["top1_frac"]), "low_coh": s["ent_med"] > THRESH["entropy_flag"]}
             for (name, a), s in sorted(RESULTS.items()) if not (a == 0.0 and name != "valence")],
  "scope_note": "Claims are about the readout method only; not a replication or refutation of "
                 "Han, Chalmers & Izmailov. Alpha is in units of OUR matched vector norm and is "
                 "not unit-converted to theirs.",
}
print("=" * 22 + " COPY-PASTE JSON BELOW " + "=" * 22)
print(json.dumps(summary, separators=(",", ":")))
print("=" * 22 + " END JSON " + "=" * 22)
print("total runtime %.1f min; ~%.2f units at the preflight rate." % (ELAPSED_MIN, GPU_RATE * ELAPSED_MIN / 60))
